In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [1]:
import requests
import json
from datetime import datetime, timezone
from pyspark.sql import Row
from pyspark.sql.functions import get_json_object, col, current_timestamp, lit, sha2, concat_ws

# ===== SETTINGS (change only here, not in other places) =====
BASE_URL = "https://hapi.fhir.org/baseR4"
RECORDS_PER_PAGE = 20
MAX_PAGES = 3   # this is our sample size, like 3 "days" of data

# Order to fetch data in, as asked in the assignment
RESOURCE_ORDER = ["Patient", "Encounter", "Observation", "Condition"]

TODAY_DATE = datetime.now().strftime("%Y-%m-%d")

StatementMeta(, ea8fbb0e-dcf2-4a82-ae27-a272e58a33c0, 3, Finished, Available, Finished, False)

In [2]:
def fetch_and_load_bronze(resource_name):
    """
    Gets data from the FHIR API, page by page.
    Saves the raw JSON as-is in the Raw layer (Files).
    Then builds a clean table for the Bronze layer.
    """
    
    first_url = BASE_URL + "/" + resource_name
    call_params = {"_count": RECORDS_PER_PAGE}
    
    data_list = []
    page_number = 1
    current_url = first_url
    
    # ---- Keep getting pages until there is no "next" page left ----
    while current_url is not None and page_number <= MAX_PAGES:
        if page_number == 1:
            response = requests.get(current_url, params=call_params)
        else:
            response = requests.get(current_url)
        
        response_data = response.json()
        extraction_time = datetime.now(timezone.utc).isoformat()
        
        data_list.append({
            "page_number": page_number,
            "extraction_timestamp": extraction_time,
            "api_url_or_params": response.url,
            "raw_data": response_data
        })
        
        # Look for the "next" link, so we know if more pages are left
        next_page_url = None
        for one_link in response_data.get("link", []):
            if one_link.get("relation") == "next":
                next_page_url = one_link.get("url")
        
        current_url = next_page_url
        page_number += 1
    
    print(resource_name, "- Pages fetched:", len(data_list))
    
    # ---- Save the raw data as-is, in a folder named by today's date ----
    raw_folder_path = "Files/raw/" + resource_name.lower() + "/" + TODAY_DATE
    for page in data_list:
        file_path = raw_folder_path + "/page_" + str(page["page_number"]) + ".json"
        mssparkutils.fs.put(file_path, json.dumps(page), overwrite=True)
    
    print(resource_name, "- Raw JSON saved to:", raw_folder_path)
    
    # ---- Turn the raw data into a proper table for Bronze layer ----
    bronze_records = []
    for page in data_list:
        entries = page["raw_data"].get("entry", [])
        for entry in entries:
            resource = entry.get("resource", {})
            bronze_records.append(Row(
                resource_id=resource.get("id", ""),
                resource_json=json.dumps(resource),
                extraction_timestamp=page["extraction_timestamp"],
                api_url_or_params=page["api_url_or_params"]
            ))
    
    bronze_df = spark.createDataFrame(bronze_records)
    table_name = "bronze_" + resource_name.lower()
    bronze_df.write.format("delta").mode("overwrite").saveAsTable(table_name)
    
    print(resource_name, "- Bronze table saved:", table_name, "|", bronze_df.count(), "records")
    print("---------------------------------------------")
    
    return bronze_df

StatementMeta(, ea8fbb0e-dcf2-4a82-ae27-a272e58a33c0, 4, Finished, Available, Finished, False)

In [3]:
print("===== BRONZE LAYER: GETTING DATA IN ORDER =====\n")
for resource in RESOURCE_ORDER:
    fetch_and_load_bronze(resource)

StatementMeta(, ea8fbb0e-dcf2-4a82-ae27-a272e58a33c0, 5, Finished, Available, Finished, False)

===== BRONZE LAYER: GETTING DATA IN ORDER =====

Patient - Pages fetched: 3
Patient - Raw JSON saved to: Files/raw/patient/2026-09-05
Patient - Bronze table saved: bronze_patient | 60 records
---------------------------------------------
Encounter - Pages fetched: 3
Encounter - Raw JSON saved to: Files/raw/encounter/2026-09-05
Encounter - Bronze table saved: bronze_encounter | 60 records
---------------------------------------------
Observation - Pages fetched: 3
Observation - Raw JSON saved to: Files/raw/observation/2026-09-05
Observation - Bronze table saved: bronze_observation | 60 records
---------------------------------------------
Condition - Pages fetched: 3
Condition - Raw JSON saved to: Files/raw/condition/2026-09-05
Condition - Bronze table saved: bronze_condition | 60 records
---------------------------------------------


In [4]:
# These are the fields we actually saw in the data for each resource
FIELD_MAPPINGS = {
    "Patient": {
        "gender": "$.gender",
        "birth_date": "$.birthDate",
        "family_name": "$.name[0].family",
        "given_name": "$.name[0].given[0]",
        "active": "$.active"
    },
    "Encounter": {
        "status": "$.status",
        "encounter_class": "$.class.code",
        "patient_reference": "$.subject.reference"
    },
    "Observation": {
        "code_text": "$.code.text",
        "patient_reference": "$.subject.reference",
        "value": "$.valueQuantity.value",
        "unit": "$.valueQuantity.unit"
    },
    "Condition": {
        "clinical_status": "$.clinicalStatus.coding[0].code",
        "verification_status": "$.verificationStatus.coding[0].code",
        "code_text": "$.code.text",
        "patient_reference": "$.subject.reference"
    }
}

def build_silver_layer(resource_name):
    """
    Takes the Bronze data, removes duplicates,
    and pulls out the important fields into their own columns.
    """
    bronze_table = "bronze_" + resource_name.lower()
    silver_table = "silver_" + resource_name.lower()
    field_mappings = FIELD_MAPPINGS[resource_name]
    
    bronze_df = spark.table(bronze_table)
    
    silver_df = bronze_df.select(
        "resource_id", "resource_json", "extraction_timestamp", "api_url_or_params"
    )
    
    # Pull each field out of the JSON text and put it in its own column
    for column_name, json_path in field_mappings.items():
        silver_df = silver_df.withColumn(column_name, get_json_object(col("resource_json"), json_path))
    
    # Remove any repeated records, keep only one copy
    silver_df = silver_df.dropDuplicates(["resource_id"])
    
    silver_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)
    
    print(resource_name, "- Silver table saved:", silver_table, "|", silver_df.count(), "clean records")
    print("---------------------------------------------")
    
    return silver_df

StatementMeta(, ea8fbb0e-dcf2-4a82-ae27-a272e58a33c0, 6, Finished, Available, Finished, False)

In [5]:
print("===== SILVER LAYER: CLEANING THE DATA =====\n")
for resource in RESOURCE_ORDER:
    build_silver_layer(resource)

StatementMeta(, ea8fbb0e-dcf2-4a82-ae27-a272e58a33c0, 7, Finished, Available, Finished, False)

===== SILVER LAYER: CLEANING THE DATA =====

Patient - Silver table saved: silver_patient | 60 clean records
---------------------------------------------
Encounter - Silver table saved: silver_encounter | 60 clean records
---------------------------------------------
Observation - Silver table saved: silver_observation | 60 clean records
---------------------------------------------
Condition - Silver table saved: silver_condition | 60 clean records
---------------------------------------------


In [6]:
def apply_scd_type2(resource_name):
    """
    Checks if any record has changed since the last time.
    If yes, keeps the old copy (marked as not current) and
    adds a new copy (marked as current). This keeps a full history.
    """
    silver_table = "silver_" + resource_name.lower()
    scd_table = "scd_" + resource_name.lower()
    
    new_data_df = spark.table(silver_table)
    
    # Make a short code (hash) from each record so we can easily check if something changed
    hash_columns = [col(c).cast("string") for c in new_data_df.columns if c != "resource_id"]
    new_data_df = new_data_df.withColumn("record_hash", sha2(concat_ws("|", *hash_columns), 256))
    
    table_exists = spark.catalog.tableExists(scd_table)
    
    if not table_exists:
        # First time we are loading this data, so mark everything as current
        scd_df = (
            new_data_df
            .withColumn("start_date", current_timestamp())
            .withColumn("end_date", lit(None).cast("timestamp"))
            .withColumn("is_current", lit(True))
        )
        scd_df.write.format("delta").mode("overwrite").saveAsTable(scd_table)
        print(resource_name, "- SCD table created (first time):", scd_df.count(), "records")
        print("---------------------------------------------")
        return
    
    # Get the current (latest) records we already have saved
    existing_df = spark.table(scd_table).filter(col("is_current") == True)
    existing_hashes = existing_df.select("resource_id", col("record_hash").alias("old_hash"))
    
    # Compare new data with old data to find what changed
    comparison_df = new_data_df.join(existing_hashes, on="resource_id", how="left")
    changed_or_new = comparison_df.filter(
        (col("old_hash").isNull()) | (col("record_hash") != col("old_hash"))
    ).drop("old_hash")
    
    if changed_or_new.count() == 0:
        print(resource_name, "- Nothing changed. SCD table stays the same.")
        print("---------------------------------------------")
        return
    
    changed_ids = [row["resource_id"] for row in changed_or_new.select("resource_id").collect()]
    full_existing = spark.table(scd_table)
    
    # Records that did not change - keep them as they are
    still_current = full_existing.filter(~col("resource_id").isin(changed_ids))
    
    # Records that changed - close them out (mark as old)
    now_inactive = (
        full_existing.filter(col("resource_id").isin(changed_ids))
        .withColumn("is_current", lit(False))
        .withColumn("end_date", current_timestamp())
    )
    
    # Add the new updated version of the changed records
    new_versions = (
        changed_or_new
        .withColumn("start_date", current_timestamp())
        .withColumn("end_date", lit(None).cast("timestamp"))
        .withColumn("is_current", lit(True))
    )
    
    # Put everything back together and save
    final_df = still_current.unionByName(now_inactive).unionByName(new_versions)
    final_df.write.format("delta").mode("overwrite").saveAsTable(scd_table)
    
    print(resource_name, "- SCD table updated:", changed_or_new.count(), "changed/new records")
    print("---------------------------------------------")

StatementMeta(, ea8fbb0e-dcf2-4a82-ae27-a272e58a33c0, 8, Finished, Available, Finished, False)

In [7]:
print("===== SCD TYPE 2: TRACKING CHANGES (FIRST LOAD) =====\n")
for resource in RESOURCE_ORDER:
    apply_scd_type2(resource)

StatementMeta(, ea8fbb0e-dcf2-4a82-ae27-a272e58a33c0, 9, Finished, Available, Finished, False)

===== SCD TYPE 2: TRACKING CHANGES (FIRST LOAD) =====

Patient - SCD table created (first time): 60 records
---------------------------------------------
Encounter - SCD table created (first time): 60 records
---------------------------------------------
Observation - SCD table created (first time): 60 records
---------------------------------------------
Condition - SCD table created (first time): 60 records
---------------------------------------------


In [8]:
from pyspark.sql.functions import count as spark_count, concat, lit as spark_lit

def build_gold_layer():
    """
    Builds a final table that is easy to use for reports -
    it shows each patient along with how many encounters,
    observations, and conditions they have.
    """
    patient_df = spark.table("silver_patient").withColumn(
        "patient_reference", concat(spark_lit("Patient/"), col("resource_id"))
    )
    encounter_df = spark.table("silver_encounter")
    observation_df = spark.table("silver_observation")
    condition_df = spark.table("silver_condition")
    
    # Count how many of each thing belongs to each patient
    encounter_counts = encounter_df.groupBy("patient_reference").agg(spark_count("resource_id").alias("total_encounters"))
    observation_counts = observation_df.groupBy("patient_reference").agg(spark_count("resource_id").alias("total_observations"))
    condition_counts = condition_df.groupBy("patient_reference").agg(spark_count("resource_id").alias("total_conditions"))
    
    # Join everything into one final table
    gold_df = (
        patient_df.select("resource_id", "given_name", "family_name", "gender", "birth_date", "patient_reference")
        .join(encounter_counts, on="patient_reference", how="left")
        .join(observation_counts, on="patient_reference", how="left")
        .join(condition_counts, on="patient_reference", how="left")
        .fillna(0, subset=["total_encounters", "total_observations", "total_conditions"])
    )
    
    gold_df.write.format("delta").mode("overwrite").saveAsTable("gold_patient_summary")
    print("Gold table 'gold_patient_summary' created |", gold_df.count(), "records")
    return gold_df

build_gold_layer()

StatementMeta(, ea8fbb0e-dcf2-4a82-ae27-a272e58a33c0, 10, Finished, Available, Finished, False)

Gold table 'gold_patient_summary' created | 60 records


DataFrame[patient_reference: string, resource_id: string, given_name: string, family_name: string, gender: string, birth_date: string, total_encounters: bigint, total_observations: bigint, total_conditions: bigint]

In [1]:
print("Testing spark session")

StatementMeta(, 032a8d19-f3ee-4d08-9171-48ff0e881502, 3, Finished, Available, Finished, False)

Testing spark session
